## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


By using nf-core we will have a fixed and reproducable way to generate the data from the same input files.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

we can use rnaseq

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

fq lint, Trim Galore!, Salmon, STAR ,RSeQC, SAMtools, Qualimap, Preseq, dupRadar, featureCounts, MultiQC, tximport, DESeq2

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [1]:
import pandas as pd

df = pd.read_excel("conditions_runs_oxy_project.xlsx")

df = df.iloc[[6, 11], :]
print(df)



   Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
6        ?  SRR23195511       x      NaN            NaN              x   
11       ?  SRR23195516       x      NaN            NaN              x   

   Genotype: SNI Genotype: Sham  
6            NaN              x  
11             x            NaN  


In [2]:
samplesheet = pd.read_csv("results/Download/samplesheet/samplesheet.csv")
samplesheet["strandedness"] = "auto"

print(samplesheet.head())

        sample                                            fastq_1  \
0  SRX19144486  ./results/Download/fastq/SRX19144486_SRR231955...   
1  SRX19144488  ./results/Download/fastq/SRX19144488_SRR231955...   

                                             fastq_2 run_accession  \
0  ./results/Download/fastq/SRX19144486_SRR231955...   SRR23195516   
1  ./results/Download/fastq/SRX19144488_SRR231955...   SRR23195511   

  experiment_accession sample_accession secondary_sample_accession  \
0          SRX19144486     SAMN32880530                SRS16557078   
1          SRX19144488     SAMN32880528                SRS16557080   

  study_accession secondary_study_accession submission_accession  ...  \
0     PRJNA926667                 SRP418759           SRA1578893  ...   
1     PRJNA926667                 SRP418759           SRA1578893  ...   

  sample_title                                   experiment_title  \
0           H2  Illumina HiSeq 2000 sequencing: GSM6958543: H2...   
1           

In [3]:
df = df.rename(columns={"Run":"run_accession"})
samplesheet = samplesheet.merge(df, on="run_accession", how="left") 
print(samplesheet)

        sample                                            fastq_1  \
0  SRX19144486  ./results/Download/fastq/SRX19144486_SRR231955...   
1  SRX19144488  ./results/Download/fastq/SRX19144488_SRR231955...   

                                             fastq_2 run_accession  \
0  ./results/Download/fastq/SRX19144486_SRR231955...   SRR23195516   
1  ./results/Download/fastq/SRX19144488_SRR231955...   SRR23195511   

  experiment_accession sample_accession secondary_sample_accession  \
0          SRX19144486     SAMN32880530                SRS16557078   
1          SRX19144488     SAMN32880528                SRS16557080   

  study_accession secondary_study_accession submission_accession  ...  \
0     PRJNA926667                 SRP418759           SRA1578893  ...   
1     PRJNA926667                 SRP418759           SRA1578893  ...   

                                        fastq_galaxy  \
0  ftp.sra.ebi.ac.uk/vol1/fastq/SRR231/016/SRR231...   
1  ftp.sra.ebi.ac.uk/vol1/fastq/SRR231

In [4]:
genotype = "Genotype: SNI"
samplesheet.loc[samplesheet[genotype] == "x", "sample"] = "SNI"
samplesheet.loc[samplesheet[genotype] != "x", "sample"] = "Sham"
samplesheet["condition"] = "oxy"
cols = ["sample", "fastq_1", "fastq_2", "strandedness", "condition"]
samplesheet = samplesheet[cols]
print(samplesheet)
samplesheet.to_csv("samplesheets/samplesheet.csv", index=False)

  sample                                            fastq_1  \
0    SNI  ./results/Download/fastq/SRX19144486_SRR231955...   
1   Sham  ./results/Download/fastq/SRX19144488_SRR231955...   

                                             fastq_2 strandedness condition  
0  ./results/Download/fastq/SRX19144486_SRR231955...         auto       oxy  
1  ./results/Download/fastq/SRX19144488_SRR231955...         auto       oxy  


In [5]:
# post here the command you used to run nf-core/rnaseq
import os.path

if not os.path.isfile("samplesheets/samplesheet_with_bams.csv"):
    print("create bams")
    !nextflow run nf-core/rnaseq -profile docker -c nextflow.config -params-file params_rnaseq_01.yaml -resume
    !cp results/initial/samplesheet/samplesheet.csv samplesheets/samplesheet_with_bams.csv
else:
    !nextflow run nf-core/rnaseq -params-file params_rnaseq_02.yaml -profile docker -c nextflow.config -resume

create bams

 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/rnaseq` [desperate_yalow] revision: a1fcdddd3b [master]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input               : samplesheets/samplesheet.csv
  outdir              : ./results/initial

Reference genome options
  genome              : GRCm38
  fasta               : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Sequence/WholeGenomeFasta/genome.fa
  gtf                 : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Annotation/Genes/genes.gtf
  gene_bed            : 

Explain all the parameters you set and why you set them in this way.



--input: input samplesheet

--save_align_intermeds: in first run we create bam files for efficient rerunning

--outdir: results directory, split in first and second run

-profile: docker containers as environments

--skip-alignment: alignment is already performed in the first run, so we just use the bam files specified in the new sample sheet.

## Browsing the results

How did the pipeline perform?

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA: not interesting for us, we want to look at genes
- Duplication
- GC content: could be contamination

What are the possible steps that could lead to poorer results?

Would you exclude any samples? If yes, which and why?

In trying to reproduce the study i wouldnt drop samples, since the study doesn't drop it. If I were the experimentator i would try to get a replacement sample

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?